# 02 · Part A: who keeps coming back to hospital?

**Question:** which diabetic patients are readmitted, and how often is each patient in hospital?

**What this notebook hands to Part C (notebook 04):** for every patient, the number of hospital stays in the past 12 months (this stay plus prior stays) and an out-of-fold readmission risk score.

1. Patient profiling in SQL (DuckDB)
2. The A1C gap: how often blood sugar was checked and acted on
3. Three readmission models (penalized logistic regression, random forest, neural network)
4. A risk score and stays-per-year for every patient

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make src/ importable

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # scikit-learn 1.9 API-change notices
import numpy as np
import pandas as pd
import duckdb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from src.config import PROCESSED, CLASS_ORDER, update_results

p = pd.read_parquet(PROCESSED / "patients.parquet")
con = duckdb.connect()
con.register("p", p)
BASE = p.readmit_30.mean()
print(f"{len(p):,} patients, {BASE:.1%} readmitted within 30 days")

## 1. Readmission rate by patient characteristic

One helper query returns, for any column: patients, % of all patients, 30-day readmission rate, and the gap to the overall rate in percentage points.

In [ ]:
def rate_by(col, order=None, min_n=300):
    q = f'''
        SELECT {col} AS level, COUNT(*) AS patients,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_patients,
               ROUND(100 * AVG(readmit_30), 1) AS readmit_pct
        FROM p GROUP BY 1 HAVING COUNT(*) >= {min_n}
    '''
    out = con.sql(q).df()
    out["vs_overall_pts"] = (out.readmit_pct - 100 * BASE).round(1)
    if order:
        out = out.set_index("level").reindex([o for o in order if o in set(out.level)]).reset_index()
    else:
        out = out.sort_values("readmit_pct", ascending=False)
    return out

p["prior_inpatient"] = pd.cut(p.number_inpatient, [-1, 0, 1, 2, 100], labels=["0", "1", "2", "3+"]).astype(str)
con.register("p", p)

dims = {
    "Prior hospital stays (past year)": rate_by("prior_inpatient", ["0", "1", "2", "3+"]),
    "Discharged to": rate_by("discharge_to"),
    "Primary diagnosis": rate_by("diag_1_group"),
}
dims["Prior hospital stays (past year)"]

In [ ]:
for name in ["Discharged to", "Primary diagnosis"]:
    print("\n" + name); display(dims[name])

**What stands out**

- **Prior hospital stays** are by far the strongest signal: patients with 3+ stays in the past year are readmitted at 26.5%, more than 3× the 8.1% for patients with none.
- **Discharge to a skilled nursing or care facility** (15.1%) carries more than twice the rate of discharge home (6.9%). These are sicker, frailer patients.
- **The primary diagnosis matters much less than I expected:** every group sits within 2 points of the 9.0% average. What predicts a return is the patient's history and where they go next, not why they came in.

## 2. The A1C gap

Guidelines say a diabetic patient admitted to hospital should get an A1C test if none was done in the past 3 months, and a high result should prompt a treatment change. How often did that happen?

In [ ]:
tested = (p.A1Cresult != "Not tested").mean()
high = p.A1Cresult.isin([">7", ">8"])
high_changed = (p.loc[high, "change"] == "Ch").mean()
print(f"A1C checked during the stay: {tested:.1%}")
print(f"High A1C patients whose diabetes meds were changed: {high_changed:.1%}")
print(f"High A1C patients left on the same meds: {1 - high_changed:.1%}")

Only **18%** of patients had their A1C checked during the stay, and **4 in 10** of those with a high result went home on unchanged diabetes medication. Patients who were tested were readmitted slightly less often than those who weren't (about 8% vs. 9%). That's a correlation, not proof testing helps, since tested patients may differ in other ways. Either way, the bigger gap is **whether anyone looked**, not which drug was chosen.

## 3. Drug classes given during the stay (model features)

One flag per class, using the shared RxClass classifier from notebook 01. Combination pills count toward each class they contain.

In [ ]:
from src.classes import classes_of     # the same classifier used for CMS drug names
DRUG_COLS = ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride", "acetohexamide", "glipizide",
             "glyburide", "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose", "miglitol", "troglitazone",
             "tolazamide", "insulin", "glyburide-metformin", "glipizide-metformin", "glimepiride-pioglitazone",
             "metformin-rosiglitazone", "metformin-pioglitazone"]
col_classes = classes_of

classes = sorted({c for col in DRUG_COLS for c in col_classes(col)}, key=CLASS_ORDER.index)
for c in classes:
    cols = [col for col in DRUG_COLS if c in col_classes(col)]
    p["on_" + c] = (p[cols] != "No").any(axis=1).astype(int)

(100 * p[["on_" + c for c in classes]].mean()).round(1).rename("% of patients")

# Readmission models

## 1. Features

Everything known **by discharge**: demographics, how the patient arrived, the stay itself (days, labs, procedures, medications), prior use of care, diagnoses, A1C and glucose results, the drug classes given, and where they were discharged to. The readmission outcome itself and IDs are excluded.

In [ ]:
y = p.readmit_30.values
p["insulin_dose_changed"] = p.insulin.isin(["Up", "Down"]).astype(int)
p["meds_changed"] = (p["change"] == "Ch").astype(int)
p["on_diabetes_meds"] = (p.diabetesMed == "Yes").astype(int)
on_cols = [c for c in p.columns if c.startswith("on_")]

NUM = ["age_mid", "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
       "has_heart_dx", "has_kidney_dx", "insulin_dose_changed", "meds_changed"] + on_cols
CAT = ["gender", "race", "admission_type", "admission_source", "discharge_to", "diag_1_group",
       "A1Cresult", "max_glu_serum", "specialty", "payer_code"]
X = p[NUM + CAT]

prep = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=200, sparse_output=False), CAT),
])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(f"train {len(X_tr):,} / test {len(X_te):,}; {len(NUM)} numeric + {len(CAT)} categorical columns")

## 2. Fit the three models

Readmissions are only ~9% of patients, so I weight the classes in training and judge the models on **ROC-AUC** (ranking quality), **PR-AUC** (precision on the rare class) and **top-decile lift**: how much more often readmission happens among the 10% of patients the model flags as highest risk. Lift is the number an outreach program actually uses.

In [ ]:
models = {
    "Penalized logistic (L1)": LogisticRegressionCV(Cs=10, penalty="l1", solver="saga", scoring="roc_auc",
                                                    class_weight="balanced", max_iter=3000, cv=5, n_jobs=-1, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=25, max_features="sqrt",
                                            class_weight="balanced_subsample", n_jobs=-1, random_state=42),
    "Neural network (MLP)": MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3, early_stopping=True,
                                          max_iter=300, random_state=42),
}

def top_decile_lift(y_true, score):
    cut = np.quantile(score, 0.9)
    return y_true[score >= cut].mean() / y_true.mean()

fitted, rows, scores = {}, [], {}
for name, m in models.items():
    pipe = Pipeline([("prep", prep), ("model", m)]).fit(X_tr, y_tr)
    s = pipe.predict_proba(X_te)[:, 1]
    fitted[name], scores[name] = pipe, s
    rows.append({"model": name, "ROC-AUC": roc_auc_score(y_te, s), "PR-AUC": average_precision_score(y_te, s),
                 "Top-decile lift": top_decile_lift(y_te, s)})
results = pd.DataFrame(rows).round(3)
results

**Reading the results.** All three models land in a narrow range (ROC-AUC 0.64–0.66), in line with published work on this dataset. The neural network is the weakest of the three, so **more flexibility doesn't buy anything here**; the random forest is best by a small margin. Readmission depends a lot on what happens after discharge, which isn't in the hospital record. Still, the 10% of patients the random forest flags are readmitted at about **2.3×** the average rate, enough to focus a follow-up program.

## 4. Score every patient for Part C

Part C (notebook 04) needs, for **every** patient: how many **prior** hospital stays they had in the past year, and what actually happened after discharge. The stay that put them in the data is **not** counted: a drug started at discharge can't prevent the stay the patient is leaving.

I also give every patient an out-of-fold risk score (5-fold, so each patient is scored by a model that never saw them) and group them into risk deciles.

In [ ]:
from sklearn.model_selection import cross_val_predict
rf_pipe = Pipeline([("prep", prep), ("model", models["Random forest"])])
p["risk_score"] = cross_val_predict(rf_pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=11),
                                    method="predict_proba", n_jobs=-1)[:, 1]
p["risk_decile"] = pd.qcut(p.risk_score.rank(method="first"), 10, labels=range(1, 11)).astype(int)
p["prior_stays"] = p.number_inpatient                 # prior stays in the past year (NOT the current stay)
p["any_readmit"] = (p.readmitted != "NO").astype(int)
print(f"Out-of-fold ROC-AUC on all patients: {roc_auc_score(y, p.risk_score):.3f}")

risk_groups = (p.groupby("risk_decile")
                 .agg(patients=("readmit_30", "size"), readmit_30_pct=("readmit_30", "mean"), any_readmit_pct=("any_readmit", "mean"),
                      prior_stays=("prior_stays", "mean"), prior_stay_pct=("number_inpatient", lambda s: (s > 0).mean()),
                      heart_pct=("has_heart_dx", "mean"), kidney_pct=("has_kidney_dx", "mean"))
                 .assign(**{c: lambda d, c=c: (100 * d[c]).round(1) for c in ["readmit_30_pct", "any_readmit_pct", "prior_stay_pct", "heart_pct", "kidney_pct"]})
                 .round({"prior_stays": 2}))
p.to_parquet(PROCESSED / "patients_scored.parquet", index=False)
risk_groups

Prior stays concentrate in the top risk deciles, and so do readmissions after discharge, both within 30 days and later. History is the strongest signal in this data; the models add only modest accuracy on top of it (AUC about 0.65).

In [ ]:
update_results("profile", {
    "overall_readmit": round(100 * BASE, 1),
    "by": {k: v[["level", "patients", "pct_of_patients", "readmit_pct"]].astype({"level": str}).to_dict("records") for k, v in dims.items()},
    "a1c_tested_pct": round(100 * tested, 1), "high_a1c_changed_pct": round(100 * high_changed, 1),
})
update_results("models", {"comparison": results.to_dict("records"), "risk_groups": risk_groups.reset_index().to_dict("records")})
print("saved")